# Lesson 15 · RNN & LSTM: networks with memory, from remembering one symbol to writing lục bát verse

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/15-rnn-en.ipynb)

This notebook accompanies [Lesson 15 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/15-rnn.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
A recurrent network reads a sequence one element at a time and carries a hidden state $h$ along. How far back can it
remember? We write an RNN, a GRU and an LSTM ourselves in numpy, measure the gradient flowing back through time on
exactly the demo's sequences, then run three language models trained on *The Tale of Kiều* (*Truyện Kiều*, Nguyễn
Du's Vietnamese poem) to see whether they can remember the 6–8 rhythm of lục bát verse (lines of 6 and 8 syllables,
alternating).

**What you'll do:**
1. Write an RNN cell and run it step by step, with the same set of weights at every step.
2. Write backpropagation through time (BPTT) yourself and check it with finite differences.
3. Measure how much of the gradient gets back to the start of the sequence: an RNN's shrinks or explodes; LSTM and
   GRU keep it thanks to their gates, but only when they are initialized correctly.
4. Train for real on the “remember the first symbol” task, then read the results of the demo's full sweep.
5. Load three character-level models trained on *The Tale of Kiều*, generate verse, and measure which network can
   remember the length of the previous line.

**Background:** [Lesson 14 · Tokens & embeddings](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-embedding/)
(a token becomes a vector) and [Lesson 09 · Vanishing gradients & initialization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/)
(products of many factors, gradient clipping). The backward pass in matrix form comes from
[Lesson 06](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/), Adam from
[Lesson 07](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/), skip connections from
[Lesson 13](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/).
**Run time:** under a minute on a CPU, including the first data download (*The Tale of Kiều*, ~150 KB, and two
results files, ~570 KB).

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. Only the long
training runs (90 for the memory task, 45 for *The Tale of Kiều*) are loaded from the very files the demo uses; the
notebook reruns part of them and checks that they give the same numbers.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign. Gradients in this lesson can be
as small as $10^{-25}$, so `p10(e)` prints $10^e$ as the demo does: an ordinary number in the range
$[10^{-3}, 10^4)$, and outside that range the form $3.7 \cdot 10^{-8}$.

In [ ]:
import hashlib
import itertools
import json
import math
import random
import re
import unicodedata
import urllib.request
from base64 import b64decode
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def p10(e):
    """10^e as in the demo: 0.23 · 38.6 · 3.7·10⁻⁸ (ordinary numbers from 0.001 to 10,000)."""
    v = 10.0**e
    if 1e-3 <= v < 1e4:
        return num(v, 4 if v < 0.01 else 3 if v < 1 else 1 if v < 100 else 0)
    k = math.floor(e)
    m = 10 ** (e - k)
    if round(m * 10) >= 100:
        m, k = m / 10, k + 1
    return f"{num(m, 1)}·10{str(k).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it
    is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(num(104204, 0), num(0.8, 1), pct(0.7835), p10(-7.43), p10(2.342), p10(-0.07))

## 2 · The data

This lesson has two data sources.

- **The Tale of Kiều** (*Truyện Kiều*) by Nguyễn Du (public domain), from the Vietnamese Wikisource: exactly
  **revision 205132**, the one the demo uses. The cell below downloads the source text (wikitext) and checks its MD5
  hash, so everyone who runs it gets exactly the same text. Wikimedia rejects Python's default program name
  (User-Agent), so the cell sets a name of its own before downloading.
- **Two results files** that the demo uses (`fetch_json`, with an MD5 check): `rnn-memory.json`, the results of the
  sweep on the “remember the first symbol” task (6 kinds of network × 5 lengths × 3 seeds), and `rnn-kieu.json`,
  three language models trained on *The Tale of Kiều* (int8 weights) along with the measurements of every training
  run. Regenerating either file takes tens of minutes on a many-core machine.

In [ ]:
KIEU_URL = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"  # revision 205132, raw wikitext
KIEU_MD5 = "d4090daac094841888ee2b2b18661065"
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # with Python's default User-Agent, Wikimedia returns error 403


def fetch_kieu(path=Path("truyen_kieu_205132.wiki")):
    if not path.exists():
        urllib.request.urlretrieve(KIEU_URL, path)
    raw = path.read_bytes()
    assert hashlib.md5(raw).hexdigest() == KIEU_MD5, f"{path} is corrupted; delete it and download it again"
    return raw


# The results of a long training run (the same file as the demo on the web). build.py replaces COURSE_DATA_URL with the
# data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


raw = fetch_kieu()
MEM = fetch_json("rnn-memory.json", "dec97dd7970ad792aee3564a845c2d39")  # memory task: the demo's sweep results
KIEU = fetch_json("rnn-kieu.json", "8f2fbe5cb77248254eb9f07770651618")  # The Tale of Kiều: models and measurements
print(f"wikitext: {num(len(raw), 0)} bytes · both results files loaded")

The wikitext has line numbers of the form `{{số|…}}` (*số*, “number”) and a few lines of description; the poem sits
between two `<poem>` tags. We clean it exactly as the demo's script does (`reel/rnn/kieu.py`) and check the MD5 hash
of the clean text. The text is read at the **character level**: every character (letters with diacritics,
punctuation marks, spaces, line breaks) is one token, as in Karpathy's (2015) models. The first 3,000 lines are for
training; the remaining lines are a validation set the model has never seen.

In [ ]:
body = raw.decode("utf-8").split("<poem>")[1].split("</poem>")[0]
lines_raw = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]  # drop the line numbers
TEXT = "\n".join(ln for ln in lines_raw if ln) + "\n"
assert hashlib.md5(TEXT.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8", "differs from the demo's text"
LINES = TEXT[:-1].split("\n")
CHARS = sorted(set(TEXT))  # the character set: 132 tokens
V = len(CHARS)
IX = {ch: i for i, ch in enumerate(CHARS)}
IDS = np.array([IX[ch] for ch in TEXT])
N_TRAIN_LINES, WARM = 3000, 512
CUT = len("\n".join(LINES[:N_TRAIN_LINES])) + 1
TRAIN, VAL = IDS[:CUT], IDS[CUT:]
VAL_LINES = LINES[N_TRAIN_LINES:]
print(f"{num(len(LINES), 0)} lines, {num(len(TEXT), 0)} characters, {V} distinct characters")
print(
    f"training: {num(N_TRAIN_LINES, 0)} lines = {num(len(TRAIN), 0)} characters · validation: {len(VAL_LINES)} lines = {num(len(VAL), 0)} characters"
)
print("first four lines:", *LINES[:4], sep="\n  ")
assert (len(LINES), len(TEXT), V) == (KIEU["source"]["lines"], KIEU["source"]["chars"], len(KIEU["chars"]))
assert "".join(CHARS) == KIEU["chars"] and VAL_LINES == KIEU["valLines"] and len(TRAIN) == KIEU["nTrain"]

## 3 · An RNN cell: reading one symbol at a time, carrying a state along

We start with the smallest task that needs memory, which is also the demo's task: **“remember the first symbol”**.
Each sequence consists of $T$ symbols drawn at random from {A, B, C, D}, then a “?”. The network must answer with the
**first** symbol. The symbols in between are just noise, from the same set of symbols as the answer, so the network
has to hold on to the first symbol for all $T$ steps. Random guessing is right 25% of the time.

A simple recurrent network (Elman, 1990) reads the symbol $x_t$ at step $t$ and updates its hidden state:

$$h_t = \tanh(x_t W_x + h_{t-1} W_h + b), \qquad h_{-1} = 0, \qquad \hat y = \operatorname{softmax}(h_T W_y + b_y)$$

- $x_t$ is the symbol's one-hot vector, so $x_t W_x$ is just one row of $W_x$: exactly the embedding of Lesson 14.
- $W_x$, $W_h$, $b$ are **the same at every step**. The number of parameters does not depend on the length of the
  sequence.
- Only the final state $h_T$ (after the “?”) is read out, through a softmax layer over 4 classes (Lesson 04).

So that the weights match the demo's **bit for bit**, we draw them exactly as the browser does: JavaScript's
`mulberry32` generator, then Box–Muller turns each pair of uniform numbers into one $\mathcal N(0, 1)$ number (as in
Lesson 09). The numbers fill $W_x$, $W_h$, $W_y$ in that order; $W_x \sim \mathcal N(0, 1/V)$,
$W_h \sim \mathcal N(0, g^2/H)$ with the factor $g = 1$ by default, $W_y \sim \mathcal N(0, 1/H)$, and the biases
are 0. The demo's seed 0 uses `mulberry32(1)`.

In [ ]:
GATES = {"rnn": 1, "gru": 3, "lstm": 4}  # number of column blocks of W: RNN 1; GRU r, z, h̃; LSTM i, f, o, c̃


def sig(z):
    """The sigmoid function σ(z), written via tanh so that it does not overflow when z is very negative."""
    return 0.5 * (1.0 + np.tanh(0.5 * z))


def mulberry32(seed, n):
    """The first n uniform numbers in [0, 1) of mulberry32(seed), the generator the demo (JavaScript) uses, in vector
    form. Every operation is 32-bit integer arithmetic (uint32 multiplication wraps around like Math.imul), so it
    matches JS bit for bit."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def normals(seed, n):
    u = mulberry32(seed, 2 * n)
    return np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller, one N(0, 1) number per pair


def init(arch, V, H, O, seed, bias=0.0, scale=1.0):
    """V one-hot inputs, H hidden units, O outputs. bias: the forget-gate bias b_f (LSTM) or the update-gate bias b_z
    (GRU); scale: the factor g of W_h. The numbers are drawn in exactly the demo's order
    (web/src/components/viz/rnn.ts)."""
    G = GATES[arch]
    z = normals(seed, V * G * H + H * G * H + H * O)
    k1, k2 = V * G * H, V * G * H + H * G * H
    p = {
        "Wx": z[:k1].reshape(V, G * H) / np.sqrt(V),
        "Wh": z[k1:k2].reshape(H, G * H) * (scale / np.sqrt(H)),
        "b": np.zeros(G * H),
        "Wy": z[k2:].reshape(H, O) / np.sqrt(H),
        "by": np.zeros(O),
    }
    if arch != "rnn":
        p["b"][H : 2 * H] = bias  # the second column block: the forget gate f (LSTM) or the update gate z (GRU)
    return p


K, H = MEM["K"], MEM["H"]  # 4 symbols A–D, 32 hidden units
SYM = "ABCD?"  # symbol number 4 is the question mark
p_rnn = init("rnn", K + 1, H, K, seed=1)
print({k: v.shape for k, v in p_rnn.items()}, "→", num(sum(v.size for v in p_rnn.values()), 0), "parameters")
print("first four integers of mulberry32(1):", (mulberry32(1, 4) * 2**32).astype(np.int64).tolist())
assert (mulberry32(1, 4) * 2**32).tolist() == [2693262067, 11749833, 2265367787, 4213581821], (
    "differs from the demo's generator"
)

The demo's gradient measurements use **64 fixed sequences**: symbols $\lfloor 4u \rfloor$ with $u$ taken from
`mulberry32(12345)`, 200 symbols per row, then cut to $T$ symbols with a “?” appended. The cell below rebuilds them,
then passes the first sequence (cut at $T = 8$) through the untrained RNN, one step at a time. The function
`step_rnn()` is one step; `forward()` is just a loop that calls it (through `STEP[arch]`) with the same set of weights.
The variable `c` is reserved for the LSTM's memory cell (section 6); the RNN just passes it along unchanged.

In [ ]:
PROBE = MEM["probe"]  # seed 12345, 64 sequences, at most 200 symbols
S_PROBE = np.floor(mulberry32(PROBE["seed"], PROBE["n"] * PROBE["maxT"]) * K).astype(int).reshape(PROBE["n"], -1)


def probe(T):
    """The demo's 64 sequences: T symbols, then a “?”; the answer is the first symbol."""
    X = np.concatenate([S_PROBE[:, :T], np.full((PROBE["n"], 1), K)], 1)
    return X, X[:, 0].copy()


def step_rnn(p, x, h, c):
    """One step: x is the symbol index (an array, one number per sequence); returns the new h, c, and what the
    backward pass needs."""
    h_new = np.tanh(p["Wx"][x] + h @ p["Wh"] + p["b"])  # p["Wx"][x] = x_t W_x with a one-hot x_t
    return h_new, c, (h, h_new)


STEP = {"rnn": step_rnn}  # section 6 adds "lstm" and "gru"


def forward(arch, p, X, h=None, c=None):
    """X: an integer array (B, T). Returns the states at every step (B, T, H), the final h and c, and the cache for
    the backward pass."""
    B, T = X.shape
    Hn = p["Wh"].shape[0]
    h = np.zeros((B, Hn)) if h is None else h
    c = np.zeros((B, Hn)) if c is None else c
    hs, cache = [], []
    for t in range(T):  # unrolled in time: the same p at every step
        h, c, e = STEP[arch](p, X[:, t], h, c)
        hs.append(h)
        cache.append(e)
    return np.stack(hs, 1), h, c, cache


X8, y8 = probe(8)
Hs, hT, _, _ = forward("rnn", p_rnn, X8[:1])
print("sequence:", " ".join(SYM[s] for s in X8[0]), " → answer", SYM[y8[0]])
for t in range(X8.shape[1]):
    print(f"step {t}: reads {SYM[X8[0, t]]}, h[:4] = {' '.join(f'{num(v, 3):>6}' for v in Hs[0, t, :4])} …")
P8 = np.exp(hT @ p_rnn["Wy"]) / np.exp(hT @ p_rnn["Wy"]).sum()
print("untrained, the network guesses:", " ".join(f"{SYM[k]} {pct(P8[0, k], 0)}" for k in range(K)))

At each step, $h$ changes with the symbol just read, but still carries part of the old $h$ along through
$h_{t-1}W_h$. Untrained, these four probabilities only reflect the random initial weights, not anything the network
has learned about the first symbol. To learn, we need the gradient of the loss with respect to $W_x$, $W_h$, $b$.

## 4 · Backpropagation through time (BPTT)

Unroll the loop in time, and an RNN is a network **$T + 1$ layers deep** in which every layer shares one set of
weights. Backpropagation on that unrolled network is called BPTT (*backpropagation through time*). Let
$\delta_t = \partial L/\partial z_t$, where $z_t = x_tW_x + h_{t-1}W_h + b$ is the value before the tanh. Going from
the last step back to the first:

$$\delta_t = \frac{\partial L}{\partial h_t} \odot (1 - h_t^2), \qquad
\frac{\partial L}{\partial h_{t-1}} \mathrel{+}= \delta_t W_h^{\top}, \qquad
\frac{\partial L}{\partial W_h} = \sum_t h_{t-1}^{\top}\,\delta_t$$

The middle formula is the key: at each step back, the gradient gets **multiplied by $W_h^{\top}$ and by
$1 - h_t^2$**. The last one is a consequence of sharing the weights: the gradient of $W_h$ is the **sum** of the
contributions of every step (likewise for $W_x$ and $b$).

The function `back_rnn()` is one step back; `backward()` calls it from $t = T$ down to $t = 0$ and accumulates. The
parameter `k` lets it stop after $k$ steps back: that is **truncated BPTT**, used in section 10. It also records
$\lVert \partial L/\partial h_t \rVert$ at every step (averaged over the batch), which section 5 will plot.

In [ ]:
def back_rnn(p, e, dh, dc):
    """One step back. In: ∂L/∂hₜ (dh). Out: δₜ = ∂L/∂zₜ, this step's part of ∂L/∂W_h, ∂L/∂hₜ₋₁, and dc (unused)."""
    h_prev, h_new = e
    dz = dh * (1 - h_new**2)  # tanh′(z) = 1 − tanh²(z)
    return dz, h_prev.T @ dz, dz @ p["Wh"].T, dc


BACK = {"rnn": back_rnn}  # section 6 adds "lstm" and "gru"


def backward(arch, p, X, cache, dHs, k=None):
    """BPTT from dHs[:, t] = ∂L/∂hₜ through the output. k: go back only k steps (truncated BPTT); None: the whole
    sequence. Returns the gradients of W_x, W_h, b and ‖∂L/∂hₜ‖ (averaged over the batch) at every step t."""
    B, T, Hn = dHs.shape
    g = {n: np.zeros_like(p[n]) for n in ("Wx", "Wh", "b")}
    dh, dc = np.zeros((B, Hn)), np.zeros((B, Hn))
    norms = np.zeros(T)
    for t in reversed(range(T)):
        if k is not None and t < T - k:
            break
        dh = dh + dHs[:, t]  # gradient reaching hₜ: via the output at step t, plus the part from all later steps
        norms[t] = np.sqrt((dh**2).sum(1)).mean()
        dz, dWh, dh, dc = BACK[arch](p, cache[t], dh, dc)
        np.add.at(g["Wx"], X[:, t], dz)  # the same W at every step: the gradient is the sum over the steps
        g["Wh"] += dWh
        g["b"] += dz.sum(0)
    return g, norms


def loss_grad(arch, p, X, y, k=None):
    """Cross-entropy of the answer y, read from the final state. Returns the loss, the accuracy, the gradients,
    ‖∂L/∂hₜ‖."""
    Hs, hT, _, cache = forward(arch, p, X)
    B = len(y)
    logits = hT @ p["Wy"] + p["by"]
    P = np.exp(logits - logits.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    loss = -np.log(P[np.arange(B), y] + 1e-300).mean()
    d = P.copy()
    d[np.arange(B), y] -= 1
    d /= B  # ∂L/∂logits = (P − onehot)/B, as in Lesson 04
    dHs = np.zeros_like(Hs)
    dHs[:, -1] = d @ p["Wy"].T  # only the last step has an output
    g, norms = backward(arch, p, X, cache, dHs, k)
    g["Wy"], g["by"] = hT.T @ d, d.sum(0)
    return loss, (P.argmax(1) == y).mean(), g, norms

A hand-written backward pass easily gets an index wrong. We check it with **finite differences** (Lesson 06): nudge
each parameter by $\pm 10^{-6}$, measure how much the loss changes, then compare with the gradient from
`backward()`. The test network is small (5 symbols, 4 hidden units, sequences of length 6) so that we can check every
parameter one by one. The cell prints the largest relative error.

In [ ]:
def gradcheck(arch, eps=1e-6):
    rng = np.random.default_rng(0)
    p = init(arch, 5, 4, 3, seed=1, bias=0.5)
    p["b"] += 0.1 * rng.standard_normal(p["b"].shape)  # nonzero biases, to check that part too
    X, y = rng.integers(0, 5, (3, 6)), rng.integers(0, 3, 3)
    g = loss_grad(arch, p, X, y)[2]
    worst = 0.0
    for name in p:
        for idx in np.ndindex(p[name].shape):
            old = p[name][idx]
            p[name][idx] = old + eps
            lp = loss_grad(arch, p, X, y)[0]
            p[name][idx] = old - eps
            lm = loss_grad(arch, p, X, y)[0]
            p[name][idx] = old
            num = (lp - lm) / (2 * eps)
            worst = max(worst, abs(num - g[name][idx]) / max(1e-8, abs(num) + abs(g[name][idx])))
    return worst


err = gradcheck("rnn")
print(f"RNN: largest relative error between BPTT and finite differences = {p10(np.log10(err))}")
assert err < 1e-5, "wrong backward pass"

An error around $10^{-9}$, right at the rounding level of finite differences: `backward()` is correct.

## 5 · How far back does the gradient get?

Now we measure on the demo's actual network: an RNN with 32 hidden units at **initialization**, the demo's 64
sequences, $T = 80$ symbols. The quantity to look at is

$$\frac{\lVert \partial L/\partial h_t \rVert}{\lVert \partial L/\partial h_T \rVert}$$

(averaged over the 64 sequences): the fraction of the gradient at the last step (the “?”) that makes it back to
step $t$. The symbol to remember is at step 0, so if the ratio at step 0 is very small, the gradient tells the network
almost nothing about what to change in order to remember it.

### 🤔 Predict first
With $T = 80$ and the factor $g = 1$ (weights $W_h \sim \mathcal N(0, 1/H)$, the “standard” initialization), roughly
how much of the gradient reaches step 0: close to 1, about 0.1, or less than $10^{-5}$?

In [ ]:
def profile(arch, p, T):
    """log10 of ‖∂L/∂hₜ‖ / ‖∂L/∂h_T‖, t = 0…T, on the demo's 64 sequences (like memory.profile)."""
    X, y = probe(T)
    norms = loss_grad(arch, p, X, y)[3]
    return np.log10(np.maximum(norms / norms[-1], 1e-300))


prof = profile("rnn", init("rnn", K + 1, H, K, seed=1), 80)
print(f"RNN, g = 1, T = 80: step 0 receives {p10(prof[0])} of the last step's gradient")
print(f"on average, per step back: ×{num(10 ** (prof[0] / 80), 3)}")
# compare with the profile memory.py saved for the demo (seed 0, the same 64 sequences): section 7 checks the rest
ref = next(c for c in MEM["checks"] if c["arch"] == "rnn" and c["scale"] == 1.0 and c["T"] == 80)
assert np.abs(prof - np.array(ref["profile"])).max() < 1e-4, "differs from the demo"

Far less than $10^{-5}$: only about $3.7 \cdot 10^{-8}$ is left. On average the gradient is multiplied by 0.808 at
each step back, and 80 such factors multiply together. This is exactly the problem of Lesson 09, except that the
“layers” here are time steps and **every layer shares one matrix** $W_h$.

The factor $g$ of $W_h$ sets the factor of each step. The left plot shows the ratio against step $t$ (log scale) for
three values of $g$: a straight line on a log scale means each step multiplies by a nearly constant factor. The right
plot sweeps $g$ from 0.25 to 3 with $T = 50$ and plots the fraction of the gradient that reaches step 0; the dashed
line is the level 1 (unchanged).

In [ ]:
GRAD_LABEL = r"$\Vert\partial L/\partial h_t\Vert \,/\, \Vert\partial L/\partial h_T\Vert$"  # y axis (mathtext)


def mt(label):
    """Labels in the figure: b_f, b_z written as subscripts."""
    return label.replace("b_f", "$b_f$").replace("b_z", "$b_z$")


fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6))
T = 80
for g, color, marker, ls in [(0.5, ORANGE, "s", ":"), (1.0, BLUE, "o", "-"), (2.0, GREEN, "^", "--")]:
    pr = profile("rnn", init("rnn", K + 1, H, K, 1, scale=g), T)
    a.plot(np.arange(T + 1), 10.0**pr, color=color, ls=ls, marker=marker, ms=4, markevery=10, label=f"g = {num(g, 1)}")
    print(f"g = {num(g, 1)}: step 0 receives {p10(pr[0])}")
a.set_yscale("log")
a.set(
    xlabel="step t (← the gradient flows back from the last step)",
    ylabel=GRAD_LABEL,
    title=f"RNN at initialization, T = {T}",
)
a.legend(frameon=False, fontsize=9)
gs = np.round(np.arange(0.25, 3.001, 0.05), 2)
g0 = np.array([profile("rnn", init("rnn", K + 1, H, K, 1, scale=g), 50)[0] for g in gs])
print(f"sweeping {len(gs)} values of g from {num(gs[0], 2)} to {num(gs[-1], 0)}")
b.plot(gs, 10.0**g0, color=BLUE, marker="o", ms=3)
b.axhline(1, color=GREY, ls="--", lw=1)
b.set_yscale("log")
b.set(xlabel="factor g of Wₕ", ylabel="fraction of the gradient reaching step 0", title="T = 50: shrink or explode?")
for ax in (a, b):
    ax.grid(alpha=0.3)
plt.tight_layout()
fmt_axes(fig)
plt.show()
g_cross = gs[np.argmax(g0 > 0)]
W1 = init("rnn", K + 1, H, K, 1)["Wh"]
print(
    f"T = 50: with g = 2, step 0 receives {p10(g0[gs == 2.0][0])}; the gradient becomes larger than at the last step from g = {num(g_cross, 2)}"
)
print(
    f"with g = 1: spectral radius of W_h = {num(np.abs(np.linalg.eigvals(W1)).max(), 2)}, largest singular value = {num(np.linalg.norm(W1, 2), 2)}"
)

With small $g$, the gradient shrinks exponentially; with large $g$, it **explodes** exponentially: with $g = 2$, what
reaches step 0 is 220 times as large as at the last step when $T = 50$, and 9,014 times when $T = 80$. Only a narrow
band around g = 1.55 keeps it nearly intact, and where that band lies also shifts with the length of the sequence and
with the weights during training. Pascanu, Mikolov & Bengio (2013) proved that if the largest singular value of
$W_h$ is less than $1/\gamma$ (with $\gamma = 1$ for tanh, the upper bound of its derivative), the gradient is sure to
vanish; for it to explode, that value *must* be greater than $1/\gamma$. At $g = 1$, $W_h$ has a largest singular
value of 1.89 and yet the gradient still shrinks: the condition for exploding is necessary, not sufficient, because
$1 - h_t^2$ is usually much smaller than 1.

Exploding gradients have a simple fix: **gradient clipping** (Lesson 09), as in every training run in this lesson
(total norm at most 1). Vanishing gradients cannot be fixed that way, because a signal that has already been drowned
out cannot be amplified. The architecture has to change.

## 6 · LSTM and GRU: the gates

A **gate** is a sigmoid vector with values in $(0, 1)$, computed from $x_t$ and $h_{t-1}$; multiplying elementwise by
it means “how much to let through”. LSTM (Hochreiter & Schmidhuber 1997; the forget gate is from Gers et al. 2000)
adds a **memory cell** $c_t$:

$$[\,i_t \;\; f_t \;\; o_t\,] = \sigma(x_t W_x + h_{t-1} W_h + b)_{\,i,f,o}, \qquad \tilde c_t = \tanh(\cdots)_{\,\tilde c}$$

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t, \qquad h_t = o_t \odot \tanh(c_t)$$

The forget gate $f_t$ decides how much of the old memory cell to keep, the input gate $i_t$ how much of the candidate
value $\tilde c_t$ to add, the output gate $o_t$ how much to let out into $h_t$. The four column blocks of $W_x$,
$W_h$, $b$ (in the order $i, f, o, \tilde c$) are computed with **one** matrix multiplication. The memory cell's path
has only multiplication by a gate, and addition:

$$\frac{\partial c_t}{\partial c_{t-1}} = f_t \quad\text{(elementwise, not counting the paths through the gates)}$$

no weight matrix, no tanh. With $f_t = 1$ it is an identity skip connection through time, like ResNet (Lesson 13).
GRU (Cho et al. 2014) is more compact: no separate memory cell, and two gates, $r_t$ (reset) and $z_t$ (update):

$$\tilde h_t = \tanh\!\big(x_t W_x + (r_t \odot h_{t-1}) W_h + b\big)_{\tilde h}, \qquad h_t = z_t \odot h_{t-1} + (1 - z_t) \odot \tilde h_t$$

in the convention of Cho et al.: $z_t$ close to 1 means **keeping** the old state (some references, such as PyTorch,
apply $r_t$ after the multiplication by $W_h$; some swap the roles of $z_t$ and $1 - z_t$). In the code, the update
gate is named `u` so that it is not confused with the pre-activation $z$. The gate's bias ($b_f$ for LSTM, $b_z$ for
GRU) is the initialization knob that section 7 will turn.

In [ ]:
def step_lstm(p, x, h, c):
    Hn = p["Wh"].shape[0]
    z = p["Wx"][x] + h @ p["Wh"] + p["b"]  # one multiplication for all four blocks: i, f, o, c̃
    i, f, o = sig(z[:, :Hn]), sig(z[:, Hn : 2 * Hn]), sig(z[:, 2 * Hn : 3 * Hn])
    g = np.tanh(z[:, 3 * Hn :])  # the candidate value c̃
    c_new = f * c + i * g  # the conveyor belt: only multiplication by a gate, and addition
    tc = np.tanh(c_new)
    return o * tc, c_new, (h, c, i, f, o, g, tc)


def step_gru(p, x, h, c):
    Hn = p["Wh"].shape[0]
    a = p["Wx"][x] + p["b"]
    zr = a[:, : 2 * Hn] + h @ p["Wh"][:, : 2 * Hn]
    r, u = sig(zr[:, :Hn]), sig(zr[:, Hn:])  # reset gate r, update gate z (named u)
    rh = r * h
    n = np.tanh(a[:, 2 * Hn :] + rh @ p["Wh"][:, 2 * Hn :])  # the candidate value h̃
    return u * h + (1 - u) * n, c, (h, r, u, rh, n)


def back_lstm(p, e, dh, dc):
    h, c, i, f, o, g, tc = e
    dcn = dc + dh * o * (1 - tc**2)  # the gradient reaching cₜ: from hₜ and from cₜ₊₁
    dz = np.concatenate([dcn * g * i * (1 - i), dcn * c * f * (1 - f), dh * tc * o * (1 - o), dcn * i * (1 - g**2)], 1)
    return dz, h.T @ dz, dz @ p["Wh"].T, dcn * f  # ∂cₜ/∂cₜ₋₁ = fₜ: the last output is the conveyor belt


def back_gru(p, e, dh, dc):
    Hn = p["Wh"].shape[0]
    h, r, u, rh, n = e
    dzn = dh * (1 - u) * (1 - n**2)
    drh = dzn @ p["Wh"][:, 2 * Hn :].T
    dzr = np.concatenate([drh * h * r * (1 - r), dh * (h - n) * u * (1 - u)], 1)
    dWh = np.concatenate([h.T @ dzr, rh.T @ dzn], 1)
    dh_prev = dh * u + drh * r + dzr @ p["Wh"][:, : 2 * Hn].T  # the straight path u ⊙ dh, then through r and the gates
    return np.concatenate([dzr, dzn], 1), dWh, dh_prev, dc


STEP.update(lstm=step_lstm, gru=step_gru)
BACK.update(lstm=back_lstm, gru=back_gru)
for arch in ("lstm", "gru"):
    err = gradcheck(arch)
    print(f"{arch.upper()}: largest relative error between BPTT and finite differences = {p10(np.log10(err))}")
    assert err < 1e-4, "wrong backward pass"
for arch in ("rnn", "gru", "lstm"):
    n = sum(v.size for v in init(arch, K + 1, H, K, 1).values())
    print(f"{arch.upper():>4}, H = {H}: {num(n, 0)} parameters")

All three backward passes match finite differences. With the same $H = 32$, the GRU has three times and the LSTM four
times as many recurrent weights as the RNN, because each gate needs its own column block.

## 7 · Gates keep the gradient, but only when they are initialized to keep it

We measure section 5's gradient ratio again for LSTM and GRU, also at initialization, on the same 64 sequences,
$T = 80$. By default every bias is 0, so at first the gates sit around $\sigma(0) = 0.5$.

### 🤔 Predict first
LSTM has a memory-cell conveyor belt along which the gradient travels with no matrix multiplication. With the
forget-gate bias $b_f = 0$, is the fraction of the gradient that reaches step 0 in an LSTM **larger or smaller** than
in a plain RNN ($3.7 \cdot 10^{-8}$)?

In [ ]:
def keep_gate(arch, p, T):
    """The average “keep” gate at each step (LSTM: fₜ, GRU: zₜ) on the demo's 64 sequences, and log10 of its product
    from t+1 to T: the part of the gradient that follows only the shortcut (the LSTM's conveyor belt, the GRU's path
    zₜ ⊙ hₜ₋₁)."""
    cache = forward(arch, p, probe(T)[0])[3]
    gate = np.array([e[3].mean() if arch == "lstm" else e[2].mean() for e in cache])
    return gate, np.r_[np.cumsum(np.log10(gate[1:])[::-1])[::-1], 0.0]


T = 80
CFG7 = [  # (architecture, gate bias, label, color, marker, line style)
    ("rnn", 0.0, "RNN, g = 1", ORANGE, "s", ":"),
    ("lstm", 0.0, "LSTM, b_f = 0", BLUE, "o", "-"),
    ("lstm", 3.0, "LSTM, b_f = 3", BLUE, "D", "--"),
    ("gru", 0.0, "GRU, b_z = 0", GREEN, "^", "-"),
    ("gru", 3.0, "GRU, b_z = 3", GREEN, "v", "--"),
]
prof7 = {}
for arch, bias, label, *_ in CFG7:
    p = init(arch, K + 1, H, K, 1, bias=bias)
    prof7[label] = profile(arch, p, T)
    line = f"{label:>14}: step 0 receives {p10(prof7[label][0]):>10}"
    if arch != "rnn":
        gate, keep = keep_gate(arch, p, T)
        line += f"   average gate {num(gate[1:].mean(), 3)}, shortcut alone: {p10(keep[0])}"
    print(line)
ratio = prof7["RNN, g = 1"][0] - prof7["LSTM, b_f = 0"][0]
print(f"the plain RNN keeps more than the LSTM with b_f = 0: {p10(ratio)} times as much")
assert prof7["LSTM, b_f = 0"][0] < prof7["RNN, g = 1"][0] < prof7["LSTM, b_f = 3"][0]

**Smaller**, by a factor of about $2 \cdot 10^5$: an LSTM with $b_f = 0$ gets only about $1.9 \cdot 10^{-13}$ of the
gradient back to step 0. The conveyor belt keeps the gradient only when the forget gate is close to 1. At
initialization $f_t \approx 0.5$, so on the conveyor belt alone the gradient loses half at every step: after 80 steps,
$0.5^{80} \approx 10^{-24}$ is left. Most of the remaining gradient goes through $h$ and the gates, that is, through
$W_h$ as in the RNN, and each of these paths adds one more sigmoid factor smaller than 1. LSTM **does not by itself
get rid of** vanishing gradients; it only gives the network a path along which the gradient *can* travel intact.
(colah's famous post calls remembering for long periods of time “practically their default behavior” for LSTMs; that
is only true when the forget gate is almost fully open.)

Add $b_f = 3$, and $f_t \approx \sigma(3) \approx 0.95$ from the start, and the gradient reaching step 0 is 0.2 times
that at the last step. GRU works the same way, although it keeps less (0.06 with $b_z = 3$): the update gate $z_t$
plays the “keep” role. That is why Gers et al. (2000) and later Jozefowicz, Zaremba & Sutskever (2015) recommend
adding 1 (or more) to the forget-gate bias; Keras sets `unit_forget_bias=True` by default, i.e. $b_f = 1$.

The figure below plots the gradient ratio against the step (log scale). The thin dotted lines of the same color are
the part that follows the shortcut alone: the product of the gates $f_t$ (LSTM) or $z_t$ (GRU) from step $t + 1$ to
the last step.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for arch, bias, label, color, marker, ls in CFG7:
    pr = prof7[label]
    ax.plot(np.arange(T + 1), 10.0**pr, color=color, ls=ls, marker=marker, ms=4, markevery=10, lw=1.8, label=mt(label))
    if arch != "rnn":
        ax.plot(
            np.arange(T + 1),
            10.0 ** keep_gate(arch, init(arch, K + 1, H, K, 1, bias=bias), T)[1],
            color=color,
            ls=":",
            lw=1,
        )
ax.set_yscale("log")
ax.set_ylim(1e-26, 3)
ax.set(xlabel="step t (← the gradient flows back from the last step)", ylabel=GRAD_LABEL)
ax.set_title("At initialization, T = 80: thick lines are the total gradient, thin dots the shortcut alone", fontsize=11)
ax.legend(frameon=False, fontsize=9, ncol=2, loc="lower right")
ax.grid(alpha=0.3)
fmt_axes(fig)
plt.show()

The thick lines of LSTM and GRU with bias 0 fall even faster than the RNN's (orange), and their thin dotted lines
(the conveyor belt alone) drop straight down to $10^{-24}$. With bias 3, both stay almost flat.

The demo computes exactly these lines in the browser. The cell below compares all of them with the demo's file: the
reference profiles (RNN with $g = 1$ and 2, GRU and LSTM with bias 0 and 3, $T$ up to 200), and the initialization
profiles of all 90 training runs of section 9 (seed $s$ starts from `mulberry32(s + 1)`).

In [ ]:
worst = 0.0
for c in MEM["checks"]:
    pr = profile(c["arch"], init(c["arch"], K + 1, H, K, 1, c.get("bias", 0.0), c.get("scale", 1.0)), c["T"])
    worst = max(worst, np.abs(pr - np.array(c["profile"])).max())
CONFIGS = {c["id"]: c for c in MEM["configs"]}
for r in MEM["runs"]:
    c = CONFIGS[r["id"]]
    for s, sd in zip(MEM["seeds"], r["seeds"]):
        pr = profile(c["arch"], init(c["arch"], K + 1, H, K, s + 1, bias=c["bias"]), r["T"])
        worst = max(worst, np.abs(pr - np.array(sd["before"])).max())  # the file stores 2 decimal places
n_prof = len(MEM["checks"]) + len(MEM["runs"]) * len(MEM["seeds"])
print(f"{n_prof} gradient profiles, largest difference from the demo's file (log10): {num(worst, 4)}")
assert worst < 0.006, "differs from the demo: check the initial weights again"

## 8 · Training for real: remembering the first symbol

The gradient at initialization is only a forecast. Now we train for real, with exactly the demo's recipe
(`reel/rnn/memory.py`):
- each step a **fresh** batch of 64 sequences, drawn from `default_rng(seed + 99)`; the test set is 1,000 fixed
  sequences from `default_rng(12345)`;
- Adam (Lesson 07) with $\eta = 0.003$, clipping the total gradient norm at 1 (Lesson 09);
- seed 0 starts from exactly the weights of sections 3–7.

The demo runs 5,000 steps; here we run 300 steps with $T = 20$ for three networks: the RNN, an LSTM with $b_f = 0$
and an LSTM with $b_f = 1$ (Keras's default), then compare with the demo's first measurement (step 250). The cell
prints the test accuracy every 50 steps.

In [ ]:
LR, CLIP, BATCH = MEM["lr"], MEM["clip"], MEM["batch"]
print(f"demo: {num(MEM['steps'], 0)} steps, batch {BATCH}, η = {num(LR, 3)}, test set {num(MEM['nTest'], 0)} sequences")


def batch(rng, T, n):
    X = rng.integers(0, K, (n, T + 1))
    X[:, -1] = K  # the “?” at the end
    return X, X[:, 0].copy()


def accuracy(arch, p, X, y):
    logits = forward(arch, p, X)[1] @ p["Wy"] + p["by"]
    return (logits.argmax(1) == y).mean()


def train_memory(arch, bias, T, seed=0, steps=300, k=None, every=50, scale=1.0):
    p = init(arch, K + 1, H, K, seed + 1, bias=bias, scale=scale)
    m = {n: np.zeros_like(v) for n, v in p.items()}
    s2 = {n: np.zeros_like(v) for n, v in p.items()}
    Xt, yt = batch(np.random.default_rng(12345), T, MEM["nTest"])
    rng = np.random.default_rng(seed + 99)
    curve = []
    for s in range(1, steps + 1):
        X, y = batch(rng, T, BATCH)
        g = loss_grad(arch, p, X, y, k)[2]
        gn = np.sqrt(sum((v**2).sum() for v in g.values()))
        if gn > CLIP:  # clip the total gradient norm
            g = {n: v * (CLIP / gn) for n, v in g.items()}
        for n in p:  # Adam
            m[n] = 0.9 * m[n] + 0.1 * g[n]
            s2[n] = 0.999 * s2[n] + 0.001 * g[n] ** 2
            p[n] -= LR * (m[n] / (1 - 0.9**s)) / (np.sqrt(s2[n] / (1 - 0.999**s)) + 1e-8)
        if s % every == 0:
            curve.append(accuracy(arch, p, Xt, yt))
    return p, np.array(curve)


live = {}
for cid, arch, bias in [("rnn", "rnn", 0.0), ("lstm0", "lstm", 0.0), ("lstm1", "lstm", 1.0)]:
    live[cid] = train_memory(arch, bias, T=20)[1]
    ref = next(r for r in MEM["runs"] if r["id"] == cid and r["T"] == 20)["seeds"][0]["curve"][0]  # step 250
    print(f"{CONFIGS[cid]['label']:>14}: " + " ".join(pct(a, 0) for a in live[cid]))
    print(f"{'':>14}  step 250: {pct(live[cid][4], 1)} (demo {pct(ref, 1)})")
    # the same initial weights, the same stream of random numbers: matches the demo (a small deviation is allowed,
    # because rounding differs between machines)
    assert abs(live[cid][4] - ref) < 0.05, "differs from the demo"

Just as the gradient at initialization predicted. The plain RNN and the LSTM with $b_f = 1$ remember perfectly after
only a few hundred steps, while the LSTM with $b_f = 0$ is stuck at the 25% of random guessing: the signal “keep the
first symbol” that reaches step 0 is too weak for the network to learn from. The three curves match the demo because
they have the same initial weights and the same batches.

## 9 · The sweep results: 6 networks × 5 lengths × 3 seeds

The demo ran the recipe above for 6 configurations (RNN; GRU with $b_z$ = 0 or 3; LSTM with $b_f$ = 0, 1 or 3), $T$
from 10 to 160, 3 seeds per cell, 5,000 steps. The figure below counts the seeds that “remember” (≥ 99% correct on
the 1,000 test sequences) in each cell: the darkness of the color and the number written in the cell both show it.

In [ ]:
Ts, cids = MEM["Ts"], [c["id"] for c in MEM["configs"]]
RUN = {(r["id"], r["T"]): r["seeds"] for r in MEM["runs"]}
solved = np.array([[sum(s["acc"] >= 0.99 for s in RUN[(c, T)]) for T in Ts] for c in cids])
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.imshow(solved, cmap=LinearSegmentedColormap.from_list("white_blue", ["#f2f2f2", BLUE]), vmin=0, vmax=3)
for i, c in enumerate(cids):
    for j, T in enumerate(Ts):
        acc = [s["acc"] for s in RUN[(c, T)]]
        ax.text(
            j,
            i,
            f"{solved[i, j]}/3",
            ha="center",
            va="center",
            fontsize=10,
            color="white" if solved[i, j] >= 2 else "black",
        )
        print(f"{CONFIGS[c]['label']:>14}, T = {T:>3}: " + " · ".join(pct(a, 1) for a in acc))
ax.set_yticks(range(len(cids)), [mt(CONFIGS[c]["label"]) for c in cids])
ax.set_xticks(range(len(Ts)), [str(T) for T in Ts])
ax.set(xlabel="sequence length T", title="Seeds that remember the first symbol (≥ 99%), 5,000 steps")
ax.spines[:].set_visible(False)
fmt_axes(fig)
plt.show()

- The **plain RNN** remembers up to $T = 20$ (2/3 seeds), then falls to random guessing from $T = 40$ on.
- **The LSTM with $b_f = 0$ is even worse than the RNN**: it only remembers at $T = 10$. With $b_f = 1$ it is on par
  with the RNN, and so is the GRU with $b_z = 0$.
- **$b_f = 3$ or $b_z = 3$** remembers up to $T = 160$ in 29 of 30 runs. The biggest difference in the figure is not
  between architectures but between two initializations of the same architecture.

“Does not remember” here means within 5,000 steps at $\eta = 0.003$: training longer or choosing a different $\eta$
might change a few cells. The demo's file also stores the gradient profiles **after** training (seed 0). The cell
below prints the fraction of the gradient that reaches step 0, before → after, for $T = 40$ and $T = 160$.

In [ ]:
for c in cids:
    row = []
    for T in (40, 160):
        s0 = RUN[(c, T)][0]
        row.append(f"T = {T:>3}: {p10(s0['before'][0]):>10} → {p10(s0['after'][0]):>10} ({pct(s0['acc'], 0)})")
    print(f"{CONFIGS[c]['label']:>14}   " + "   ".join(row))

In the networks that learned, the gradient reaching step 0 is still large after training (for the GRU with
$b_z = 3$ it even rises to 1.1 and 2.8): the network has tuned its gates to keep it. In the networks that failed, the
gradient is even **smaller** than at the start, at times down to $10^{-44}$: getting no signal from step 0, the
network learns only what is in the nearby steps, and the path back into the past fades even more. It is a vicious
circle: to learn how to remember, the gradient has to reach the place to be remembered from the very start.

## 10 · Truncated BPTT

With very long sequences (a book, an hour of audio), backpropagating through the whole sequence takes memory
proportional to its length. In practice people use **truncated BPTT**: the state is still carried forward through
the whole sequence, but the gradient only flows back $k$ steps. The parameter `k` of `backward()` does exactly that.

### 🤔 Predict first
The RNN of section 8 learned the $T = 20$ task with full BPTT. If the gradient only flows back $k = 10$ steps, what
percentage does it get right after 300 training steps: still close to 100%, about half, or around 25%?

In [ ]:
rnn_k10 = train_memory("rnn", 0.0, T=20, k=10)[1]
print("RNN, T = 20, k = 10: " + " ".join(pct(a, 0) for a in rnn_k10))
print("RNN, T = 20, full:   " + " ".join(pct(a, 0) for a in live["rnn"]))
assert rnn_k10.max() < 0.35 < live["rnn"][-1]
lstm3_k10 = train_memory("lstm", 3.0, T=20, k=10)[1]
print("LSTM, b_f = 3, T = 20, k = 10: " + " ".join(pct(a, 0) for a in lstm3_k10))
assert 0.4 < lstm3_k10[-1] < 0.95

Around 25%: random guessing. With $k = 10$, the gradient never reaches step 0, so the network has **no way** of
knowing that the first symbol matters. Truncated BPTT sets a hard limit: the network cannot *learn* dependencies
longer than $k$ steps.

Yet the LSTM with $b_f = 3$, also with only $k = 10$, gets to about 78%. Why? With $f_t \approx 0.95$ from the start,
its memory cell *already* keeps part of the first symbol until the end of the sequence, before it has learned
anything. The readout layer $W_y$ only needs the gradient at the last step, so it can learn to exploit that
ready-made memory. Truncation blocks learning *how to remember*; it does not block using the memory that the state
happens to carry along. (Karpathy, 2015, trained a character model on Paul Graham's essays with BPTT truncated at
100 characters.)

## 11 · A character-level language model of *The Tale of Kiều*

The real task: read *The Tale of Kiều* **one character at a time** and predict the next character. There is an
output at *every* step, not just the last one, and the loss is the average cross-entropy. The demo
(`reel/rnn/kieu.py`) trained three kinds of network, $H = 128$, 3 seeds each: Adam, 4,000 steps, each step 32 windows
of 100 characters, gradient clipping at 1, the LSTM initialized with $b_f = 1$; $\eta$ chosen from
{0.001, 0.002, 0.004} on the validation set, keeping the checkpoint with the lowest validation loss. That takes tens
of minutes on a many-core machine, so we load the **weights of seed 0** that the demo downloads. They are stored as
int8: one scale factor $s$ per column of each matrix, and $W = q \cdot s$ with $q \in \{-127, \dots, 127\}$. Every
number in the demo is computed from these very rounded weights.

In [ ]:
def dequant(d):
    """int8 (base64) with one scale factor per column → a matrix of floats, exactly as the demo unpacks it."""
    q = np.frombuffer(b64decode(d["q"]), np.int8).reshape(d["rows"], d["cols"])
    return q * np.array(d["s"])


LM = {}
for arch, d in KIEU["ship"].items():
    LM[arch] = {k: dequant(d[k]) for k in ("Wx", "Wh", "Wy")} | {k: np.array(d[k], float) for k in ("b", "by")}
MODELS = {(m["arch"], m["H"]): m for m in KIEU["models"]}
print(
    "η tried:",
    " · ".join(num(lr, 3) for lr in KIEU["lrs"]),
    f"· {num(KIEU['steps'], 0)} steps, batch {KIEU['batch']} × {KIEU['window']} characters",
)
for arch, p in LM.items():
    n = sum(v.size for v in p.values())
    print(f"{arch.upper():>4}, H = 128: {num(n, 0):>7} parameters, chosen η = {num(MODELS[(arch, 128)]['lr'], 3)}")
    assert n == MODELS[(arch, 128)]["params"]

We let the three networks read the last four lines of the training set, then compare the logits of the next
character with the numbers the demo saved. A trained network is stable, so the difference is only the machine's
rounding. (An untrained RNN can be *chaotic*: rounding errors get amplified exponentially from step to step, and this
comparison would not hold up.)

In [ ]:
def run_text(arch, p, text, h=None, c=None):
    """Read text one character at a time; return the final h and c."""
    h = np.zeros((1, 128)) if h is None else h
    c = np.zeros((1, 128)) if c is None else c
    for ch in text:
        h, c, _ = STEP[arch](p, np.array([IX[ch]]), h, c)
    return h, c


PROMPT = KIEU["prompt"]
print(PROMPT)
for arch, p in LM.items():
    h, _ = run_text(arch, p, PROMPT)
    logits = (h @ p["Wy"] + p["by"])[0]
    err = np.abs(logits - np.array(KIEU["ship"][arch]["check"])).max()
    top = np.argsort(-logits)[:3]
    P = np.exp(logits - logits.max()) / np.exp(logits - logits.max()).sum()
    print(
        f"{arch.upper():>4}: first character of the new line, top 3 choices: "
        + ", ".join(f"“{CHARS[i]}” {pct(P[i], 1)}" for i in top)
    )
    print(f"      largest logit difference from the demo: {p10(np.log10(err))}")
    assert err < 1e-6, "differs from the demo"

The logits match to $10^{-8}$: these are exactly the demo's three networks. All three predict that the new line
starts with a capital letter, just as the text habitually does.

## 12 · How well does the network predict the next character?

We measure in **bits per character** on the validation set: the average $-\log_2 p(\text{correct character})$,
exactly the demo's measure. Guessing uniformly over 132 characters costs $\log_2 132 \approx 7.04$ bits; using only
character frequencies (remembering nothing) costs 5.18 bits. The network reads the last 512 characters of the
training set to “warm up” its state, then reads the 254 validation lines in one go, carrying the state all the way.

In that same reading pass, we keep the state and the logits at every character: sections 14 and 15 reuse them.

In [ ]:
def read_val(arch, p):
    """Warm up on 512 training characters, then read the validation set one character at a time. Returns, after each
    validation character: the logits of the next character, the state h, and the memory cell c (LSTM)."""
    h, c = run_text(arch, p, "".join(CHARS[i] for i in TRAIN[-WARM:]))
    logits, hs, cs = [], [], []
    for x in VAL:
        h, c, _ = STEP[arch](p, np.array([x]), h, c)
        logits.append((h @ p["Wy"] + p["by"])[0])
        hs.append(h[0])
        cs.append(c[0])
    return np.array(logits), np.array(hs), np.array(cs)


def log_softmax(z):
    z = z - z.max(-1, keepdims=True)
    return z - np.log(np.exp(z).sum(-1, keepdims=True))


counts = np.bincount(TRAIN, minlength=V) + 1  # character frequencies in the training set (+1 so no probability is 0)
unigram = -np.log2(counts / counts.sum())[VAL[1:]].mean()
print(
    f"uniform guess: {num(np.log2(V), 2)} bits/character · character frequencies only: {num(unigram, 2)} bits/character"
)
READ = {arch: read_val(arch, p) for arch, p in LM.items()}
for arch, (lg, _, _) in READ.items():
    bits = -log_softmax(lg[:-1])[np.arange(len(VAL) - 1), VAL[1:]].mean() / np.log(2)
    ref = MODELS[(arch, 128)]["seeds"][0]["bits"]
    print(f"{arch.upper():>4}: {num(bits, 4)} bits/character on the validation set (demo: {num(ref, 4)})")
    assert abs(bits - ref) < 1e-3, "differs from the demo"
    steps = [sd["step"] for sd in MODELS[(arch, 128)]["seeds"]]
    last = MODELS[(arch, 128)]["seeds"][0]["curve"][-1]  # [bits/character on training, on validation] at step 4,000
    print(f"      best checkpoint of the 3 seeds at step {', '.join(num(v, 0) for v in steps)}", end="")
    print(f"; at step 4,000 (seed 0): training {num(last[0], 2)}, validation {num(last[1], 2)}")

The LSTM predicts best, then the GRU, then the RNN; all three are far better than the “character frequencies only”
level. *The Tale of Kiều* has only 104,204 characters, so the models are small and learn fast: the best checkpoint on
the validation set comes early, before the halfway point of the 4,000 steps (the cell above prints each seed's step).
After that the network starts to **memorize** the training set (Lesson 08): by step 4,000, the bits per character on
the training set are still falling, while on the validation set they have risen again.

## 13 · Generating verse one character at a time

Generating text is reading turned around: take the distribution of the next character, **sample a character** from
it, have the network read that very character, and repeat. The **temperature** $\tau$ divides the logits before the
softmax, $p \propto \exp(z/\tau)$: $\tau < 1$ makes the distribution sharper (safe, prone to repetition), $\tau > 1$
makes it flatter (varied, with many spelling errors). The demo uses $\tau = 0.8$, starts from the last four lines of
the training set, and draws its random numbers with `default_rng(0)` for seed 0.

To score the verse, we need the rules of **lục bát** (“six–eight”) verse: a 6-syllable line, then an 8-syllable
line, alternating; syllable 2 has a level tone (*bằng*: ngang, as in “a”, and huyền, “à”), syllable 4 an oblique
tone (*trắc*: sắc “á”, hỏi “ả”, ngã “ã”, nặng “ạ”), syllable 6 a level tone, and the 8-syllable line adds a
level-toned syllable 8. Vietnamese writes each syllable as a separate word, so the syllables of a line are the words
between its spaces. The tone can be read from the diacritics: decomposing the letters with Unicode's NFD
normalization turns the sắc, hỏi, ngã and nặng marks into separate combining characters. (We do not check the
rhyme.)

In [ ]:
TRAC = {"́", "̉", "̃", "̣"}  # the sắc, hỏi, ngã, nặng marks after NFD decomposition; ngang and huyền are level (bằng)


def syllables(line):
    return re.sub(r"[^\w\s]", " ", line).split()  # drop punctuation, split on whitespace


def trac(s):
    return any(ch in TRAC for ch in unicodedata.normalize("NFD", s.lower()))


def tone_ok(w):
    """Syllable 2 level, 4 oblique, 6 level (and 8 level in an 8-syllable line)."""
    return len(w) in (6, 8) and not trac(w[1]) and trac(w[3]) and not trac(w[5]) and (len(w) == 6 or not trac(w[7]))


def tone_text(w):
    return " ".join("T" if trac(w[k]) else "B" for k in (1, 3, 5, 7) if k < len(w)) if len(w) in (6, 8) else "–"


def stats(lines):
    """Fractions of lines with 6 or 8 syllables, of pairs of consecutive lines that alternate 6↔8, of lines that
    follow the level/oblique tone rule, and of lines copied verbatim."""
    n = [len(syllables(ln)) for ln in lines]
    p6, p8 = np.mean([k == 6 for k in n]), np.mean([k == 8 for k in n])
    return {
        "n68": np.mean([k in (6, 8) for k in n]),
        "alt": np.mean([(a, b) in ((6, 8), (8, 6)) for a, b in itertools.pairwise(n)]),
        "chance": 2 * p6 * p8,  # alternation by chance, if each line's length were chosen independently
        "tone": np.mean([tone_ok(syllables(ln)) for ln, k in zip(lines, n) if k in (6, 8)]),
        "copied": np.mean([ln in CORPUS for ln in lines]),
    }


CORPUS = set(LINES)
S = list(
    itertools.chain.from_iterable(syllables(ln) for ln in LINES)
)  # the whole poem's syllables, shuffled as in the demo
random.Random(0).shuffle(S)
it = iter(S)
shuffled = [" ".join(next(it) for _ in range(6 if i % 2 == 0 else 8)) for i in range(len(LINES))]
real, shuf = stats(VAL_LINES), stats(shuffled)
print(
    f"254 real validation lines: {pct(real['n68'])} with 6/8 syllables, {pct(real['alt'])} alternating, {pct(real['tone'])} following the tone rule"
)
print(f"the whole poem's syllables shuffled, cut back into 6/8 lines: {pct(shuf['tone'])} following the tone rule")
assert round(real["tone"], 4) == KIEU["real"]["tone"] and round(shuf["tone"], 4) == KIEU["shuffled"]["tone"]

The real verse of the validation set: every line has 6 or 8 syllables, 100% of them alternate, and almost every line
follows the level/oblique tone rule. Shuffle the syllables of the whole poem and cut them back into 6/8 lines, and
only about one line in ten follows the rule: that is the chance level of the tone rule.

Now each network generates 20,000 characters (about 600 lines) exactly as in the demo, and we score them. The cell
prints the first 6 lines of each network, with their number of syllables and the tones at positions 2‑4‑6(‑8)
(B = *bằng*, level; T = *trắc*, oblique).

In [ ]:
def generate(arch, p, prompt, n, temp, rng):
    h, c = run_text(arch, p, prompt)
    out = []
    for _ in range(n):
        z = (h @ p["Wy"] + p["by"])[0] / temp  # divide the logits by the temperature
        pr = np.exp(z - z.max())
        x = rng.choice(V, p=pr / pr.sum())  # sample a character from the distribution
        out.append(CHARS[x])
        h, c, _ = STEP[arch](p, np.array([x]), h, c)  # the network reads the very character it just generated
    return "".join(out)


TEMP, N_SAMPLE = KIEU["temp"], KIEU["nSample"]
print(f"each network generates {num(N_SAMPLE, 0)} characters at temperature {num(TEMP, 1)}")
GEN = {}
for arch, p in LM.items():
    text = generate(arch, p, PROMPT, N_SAMPLE, TEMP, np.random.default_rng(0))
    GEN[arch] = text.split("\n")[:-1]  # the last line is unfinished
    print(f"── {arch.upper()} (temperature {num(TEMP, 1)}, seed 0)")
    for ln in GEN[arch][:6]:
        w = syllables(ln)
        print(f"  {ln:<44} {len(w)} syllables  {tone_text(w)}")
    assert GEN[arch][:8] == MODELS[(arch, 128)]["seeds"][0]["sample"], "differs from the demo"

It looks like Vietnamese, mostly correctly spelled, often with a comma at the end of the 6-syllable line and a period
at the end of the 8-syllable line as in the original, but it carries no meaning. The first eight lines of each
network match the samples the demo saved word for word. Scoring all 20,000 characters:

In [ ]:
print(
    f"{'':>5} {'lines':>5} {'6/8 syllables':>13} {'alternate 6↔8':>13} {'chance 2·p₆·p₈':>15} {'tone rule':>10} {'copied':>7}"
)
for arch in LM:
    st = stats(GEN[arch])
    ref = MODELS[(arch, 128)]["seeds"][0]["gen"]
    print(
        f"{arch.upper():>5} {len(GEN[arch]):>5} {pct(st['n68'], 1):>13} {pct(st['alt'], 1):>13} {pct(st['chance'], 1):>15}"
        f" {pct(st['tone'], 1):>10} {pct(st['copied'], 1):>7}"
    )
    for key in ("n68", "alt", "tone", "copied"):  # a small deviation is allowed: one different draw changes the rest
        assert abs(st[key] - ref[key]) < 0.02, f"{key} differs from the demo"

All three networks learned the **level/oblique tone** rule (90.5–94.0% of lines follow it, against 10.79% for the
shuffled syllables): that is a local matter, within a few syllables of one line. Not a single line is copied verbatim
from *The Tale of Kiều*.

The **6↔8 alternation** figure needs careful reading. A network that remembers nothing about the previous line, and
just picks each line's length by its own proportions ($p_6$ of 6-syllable and $p_8$ of 8-syllable lines), still
alternates correctly with probability $2p_6p_8$, by chance. At seed 0, the LSTM beats that level by about 13
percentage points, the GRU by about 5 points, while the RNN does not beat it.

### 🤔 Predict first
The numbers above are for seed 0 only. Across the demo's 3 seeds, does the LSTM always remember the 6–8 rhythm better
than the GRU?

In [ ]:
LABEL = {
    ("lstm", 128): "LSTM",
    ("gru", 128): "GRU",
    ("rnn", 128): "RNN",
    ("gru", 150): "GRU 150",
    ("rnn", 280): "RNN 280",
}


def chance_alt(g):
    """2·p₆·p₈ from the histogram of syllable counts that the demo saved for each sample."""
    return 2 * g["hist"][6] * g["hist"][8] / g["lines"] ** 2


print(f"{'':>8} {'params':>8}   6↔8 alternation above chance (percentage points), seed 0 · 1 · 2")
for key, m in MODELS.items():
    gain = [100 * (s["gen"]["alt"] - chance_alt(s["gen"])) for s in m["seeds"]]
    print(f"{LABEL[key]:>8} {num(m['params'], 0):>8}   " + " · ".join(f"{num(v, 1):>5}" for v in gain))

**No.** The GRU's seed 1 beats chance by 28.7 points, more than any seed of the LSTM; the LSTM's seed 2 beats it by
only 0.3 points, which means it remembers almost nothing. The sampling results swing widely between seeds, because
600 randomly generated lines are a noisy measurement, and because each training run finds (or does not find) a way to
remember. With $\eta$ chosen separately for each kind of network on the validation set, the GRU is **no worse** than
the LSTM on this measure. Chung et al. (2014) also found GRU and LSTM comparable, with no clear winner; Jozefowicz et
al. (2015) found that the LSTM catches up with the GRU when 1 is added to $b_f$.

The plain RNN is consistent in one respect: none of its seeds beats chance by more than 4 points. To see this more
clearly, we need a measurement that is less noisy than sampling.

## 14 · A less noisy test: end the line or continue?

Reading a **real** line of the validation set (the network is fed the true characters; it does not generate them),
at the end of the 6th syllable the network has to choose: **end the line** (a 6-syllable line) or **continue** with a
space (an 8-syllable line). The first six syllables of both kinds of line follow the same rule, so there is only one
way to guess right: remember how long the *previous* line was. The validation set has exactly 127 lines of each kind,
so random guessing is right 50% of the time. The cell below measures how far the information to remember lies from
the decision point. (In training, each 100-character window starts from $h = 0$, so the gradient flows back at most
100 characters: enough to learn this distance.)

In [ ]:
def positions(line):
    """For each character: the number (1, 2, …) of the syllable that contains it within its line; 0 for a line
    break."""
    pos, k, inword = [], 0, False
    for ch in line:
        if ch == "\n":
            pos.append(0)
            k, inword = 0, False
            continue
        word = ch.isalnum()
        if word and not inword:
            k += 1
        inword = word
        pos.append(k)
    return np.array(pos)


LAST6 = [max(i for i, k in enumerate(positions(ln)) if k == 6 and ln[i].isalnum()) for ln in VAL_LINES]
N_SYL = np.array([len(syllables(ln)) for ln in VAL_LINES])
end6 = [ln[-1] for ln in LINES if len(syllables(ln)) == 6]
gap = np.array(LAST6) + 2  # characters from the punctuation mark ending the previous line to the decision point
print(
    f"validation set: {(N_SYL == 6).sum()} 6-syllable and {(N_SYL == 8).sum()} 8-syllable lines → random guessing gets {pct(0.5, 0)}"
)
print(f"whole poem: {pct(np.mean([e == ',' for e in end6]), 1)} of 6-syllable lines end with a comma")
print(
    f"the previous line's punctuation mark is {num(gap.mean(), 1)} characters from the decision point (from {gap.min()} to {gap.max()})"
)

A 6-syllable line almost always ends with a comma, an 8-syllable line with a period, an exclamation mark or a
question mark. So what has to be remembered is just **one bit** (was the previous line a 6- or an 8-syllable line?),
but it has to be held across about 26 characters, close to the $T = 20$–40 tasks of section 9. The cell below
computes the test from the very logits collected in section 12, for seed 0 of the three networks, and compares with
the demo.

In [ ]:
DECIDE = {}
for arch, (lg, _, _) in READ.items():
    start = np.cumsum([0] + [len(ln) + 1 for ln in VAL_LINES[:-1]])  # position of each line's first character in VAL
    p_space = np.exp(log_softmax(lg[start + np.array(LAST6)])[:, IX[" "]])  # P(space): the network chooses to continue
    right = np.where(N_SYL == 8, p_space > 0.5, p_space < 0.5)
    DECIDE[arch] = right.mean()
    ref = MODELS[(arch, 128)]["seeds"][0]["decide"]
    print(
        f"{arch.upper():>4}: right on {pct(right.mean())} of lines (demo {pct(ref['acc'])}); average P(continue): "
        f"6-syllable lines {num(p_space[N_SYL == 6].mean(), 3)}, 8-syllable lines {num(p_space[N_SYL == 8].mean(), 3)}"
    )
    assert abs(right.mean() - ref["acc"]) < 0.005 and abs(p_space[N_SYL == 6].mean() - ref["space6"]) < 1e-3

At seed 0, the LSTM chooses right on 78.35% of lines, the GRU on 65.35%, and the RNN on 55.91%, close to random
guessing. On 6-syllable lines, the RNN still gives a probability of continuing of almost one half, which means it does
not know how long the previous line was. This test uses real lines, so it has no sampling noise; the noise that
remains comes from the training run itself. The figure below shows all 3 seeds of every network the demo trained,
including a larger GRU and RNN with about as many parameters as the LSTM (150 and 280 hidden units). Each dot is one
seed; the gray line is the 50% of random guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
STYLE = {"lstm": (BLUE, "o"), "gru": (GREEN, "^"), "rnn": (ORANGE, "s")}
keys = list(MODELS)
for i, key in enumerate(keys):
    color, marker = STYLE[key[0]]
    acc = [s["decide"]["acc"] for s in MODELS[key]["seeds"]]
    ax.scatter(100 * np.array(acc), [i] * 3, color=color, marker=marker, s=60, zorder=3, edgecolor="white")
    bits = " · ".join(num(s["bits"], 3) for s in MODELS[key]["seeds"])
    print(
        f"{LABEL[key]:>8} ({num(MODELS[key]['params'], 0)} parameters): right " + " · ".join(pct(a, 1) for a in acc),
        f"  bits/character {bits}",
    )
ax.axvline(50, color=GREY, ls="--", lw=1)
ax.set_yticks(
    range(len(keys)), [f"{LABEL[k]} ({num(MODELS[k]['params'] / 1000, 0)} thousand parameters)" for k in keys]
)
ax.invert_yaxis()
ax.set(xlim=(45, 95), xlabel="right after syllable 6 (%), 254 validation lines")
ax.set_title("Does it remember how long the previous line was? Each dot is one seed", fontsize=11)
ax.grid(axis="x", alpha=0.3)
fmt_axes(fig)
plt.show()
rnn_all = [s["decide"]["acc"] for k, m in MODELS.items() if k[0] == "rnn" for s in m["seeds"]]
gated = [s["decide"]["acc"] for k, m in MODELS.items() if k[0] != "rnn" for s in m["seeds"]]
clear = max(rnn_all) + 0.05
print(f"6 RNN seeds: {pct(min(rnn_all), 1)}–{pct(max(rnn_all), 1)}")
print(
    f"gated networks clearly above every RNN seed (above {pct(clear, 1)}): {sum(a > clear for a in gated)}/{len(gated)} seeds"
)

This is the lesson's robust result:
- **All 6 plain-RNN seeds** sit close to random guessing (52.4–57.9%), including the 280-unit RNN, which has as many
  parameters as the LSTM. More hidden units do not give the RNN a longer memory; what it lacks is a path for the
  gradient to cross 26 characters.
- **Gated networks** *can* remember: 5 of 9 seeds clearly beat every RNN seed, the best reaching 89.4%. But the other
  4 seeds are on par with the RNN. Gates make long-range memory **learnable**; they do not guarantee that every
  training run learns it.
- Between LSTM and GRU, the difference lies **within the variation between seeds**.

Bits per character (the last column) rank more consistently: for the same seed, the LSTM is always lower than the
GRU, and every RNN seed is higher than every gated network (the 280-unit RNN is even worse than the 128-unit one).
This measure consists mostly of local predictions (spelling, words that often go together), so it says little about
long-range memory.

## 15 · Looking inside one unit: the network counts syllables

To know when to end the line, the network has to know which syllable it is on. Karpathy, Johnson & Fei-Fei (2015)
found memory cells in LSTMs that track line length, quotation marks and brackets. We run the demo's search: in the
states collected in section 12 (the LSTM's memory cell $c_t$, the other two networks' $h_t$), we look for the unit
with the largest **correlation** (Pearson coefficient $r$) with the syllable's position in the line.

In [ ]:
pos_val = positions("".join(CHARS[i] for i in VAL))
keep = pos_val > 0  # drop the line-break characters
TRACK = {}
for arch, (_, hs, cs) in READ.items():
    st = cs if arch == "lstm" else hs
    Z = st[keep] - st[keep].mean(0)
    y = pos_val[keep] - pos_val[keep].mean()
    r = Z.T @ y / np.sqrt((Z**2).sum(0) * (y**2).sum())
    unit = int(np.argmax(np.abs(r)))
    TRACK[arch] = (unit, st[:, unit], r[unit])
    ref = KIEU["ship"][arch]["tracker"]
    print(
        f"{arch.upper():>4}: unit #{unit} of {'c' if arch == 'lstm' else 'h'}, r = {num(r[unit], 3)} (demo: #{ref['unit']}, r = {num(ref['r'], 3)})"
    )
    assert unit == ref["unit"] and abs(r[unit] - ref["r"]) < 1e-3, "differs from the demo"

All three match the demo. The left plot shows unit #51 of the LSTM's memory cell as it reads the first four
validation lines, character by character (gray dashed lines: line breaks). The right plot standardizes the unit found
in each network (subtract the mean, divide by the standard deviation, flip the sign if $r < 0$), then averages it by
syllable position over the whole validation set; for a network that counts well, the line rises steadily.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [3, 2]})
unit, trace, _ = TRACK["lstm"]
n_show = sum(len(ln) + 1 for ln in VAL_LINES[:4])
a.plot(np.arange(n_show), trace[:n_show], color=BLUE, lw=1.5)
for i in np.flatnonzero(VAL[:n_show] == IX["\n"]):
    a.axvline(i, color=GREY, ls="--", lw=1)
a.set(
    xlabel="character number … (first 4 validation lines)",
    ylabel=f"c[{unit}]",
    title=f"LSTM: memory cell #{unit}, reading character by character",
)
for arch, (unit, trace, r) in TRACK.items():
    zs = np.sign(r) * (trace - trace[keep].mean()) / trace[keep].std()
    mean = [zs[pos_val == k].mean() for k in range(1, 9)]
    color, marker = STYLE[arch]
    b.plot(range(1, 9), mean, color=color, marker=marker, label=f"{arch.upper()} #{unit} (|r| = {num(abs(r), 2)})")
b.set(
    xlabel="syllable number … in the line", ylabel="standardized value", title="Average over the 254 validation lines"
)
b.legend(frameon=False, fontsize=9)
for ax in (a, b):
    ax.grid(alpha=0.3)
plt.tight_layout()
fmt_axes(fig)
plt.show()
print(*VAL_LINES[:4], sep="\n")
line_len = np.concatenate([[len(syllables(ln))] * (len(ln) + 1) for ln in VAL_LINES])  # syllables of each char's line
for arch, (unit, trace, r) in TRACK.items():
    zs = np.sign(r) * (trace - trace[keep].mean()) / trace[keep].std()
    at = {n: [zs[(pos_val == k) & (line_len == n)].mean() for k in (1, 6)] for n in (6, 8)}
    print(
        f"{arch.upper():>4} #{unit}: syllable 1 → 6, 6-syllable lines: {num(at[6][0], 2)} → {num(at[6][1], 2)};"
        f" 8-syllable lines: {num(at[8][0], 2)} → {num(at[8][1], 2)}"
    )
    if arch == "lstm":  # sign-flipped: lower means a higher cell value
        assert at[8][0] < at[6][0] and at[8][1] < at[6][1], "at syllables 1 and 6, the 8-line cell is no longer higher"

The LSTM's memory cell #51 falls steadily with each syllable, then jumps back up at every line break: a **syllable
counter** that nobody programmed, which formed by itself while the network learned to predict characters. In the
left plot it dips lower on the 8-syllable lines (lines 2 and 4) than on the 6-syllable ones, but not because it falls
faster: it starts higher, and dips lower only because it keeps counting through syllables 7 and 8. The LSTM's printed
line shows this: the standardized values there are sign-flipped (lower means a higher cell value), and on 8-syllable
lines they are lower than on 6-syllable lines at syllable 1 as well as at syllable 6. Already at syllable 1,
8-syllable lines differ from 6-syllable ones, which means the cell carries along what it remembered from the
previous line. The GRU's unit counts almost as well.
The RNN's best unit is clearly worse ($|r|$ = 0.66): it jumps once at syllable 4 and then nearly stands still, and it
barely distinguishes 6-syllable from 8-syllable lines. (This is the *most correlated* unit; that does not mean the
network uses only this one.)

## 16 · Try changing things

The cells below rerun the same functions with different parameters. You can change the values and run them again.

**Can initialization rescue the plain RNN too?** Section 5 showed that the factor $g$ of $W_h$ drastically changes the
fraction of the gradient that reaches step 0. Train the RNN on $T = 40$ (a length at which it fails in all 3 of the
demo's seeds) with a few values of $g$, seed 0, 300 steps:

In [ ]:
for g in (1.0, 1.5, 2.0):
    acc = train_memory("rnn", 0.0, T=40, scale=g)[1]
    print(f"RNN, T = 40, g = {num(g, 1)}: " + " ".join(pct(v, 0) for v in acc))

With $g = 1.5$, near the boundary between shrinking and exploding (section 5), the plain RNN learns $T = 40$ after
only a few hundred steps; with $g = 2$ it learns too, more slowly: the gradient explodes, and clipping keeps each
update from being too large. Initializing near that boundary is the idea behind orthogonal or identity
initialization for RNNs ([Le, Jaitly & Hinton 2015](https://arxiv.org/abs/1504.00941) set $W_h = I$ for RNNs that use
ReLU). Initialization only chooses the starting point; gates let the network *learn for itself*, symbol by symbol,
when to keep and when to forget. Try $T = 80$ as well, or other seeds (`seed=1`).

**Temperature and the opening line.** Change `temp` and `prompt` (use only characters that occur in *The Tale of
Kiều*). The opening line below is the first line of the poem, which is in the training set.

In [ ]:
prompt = "Trăm năm trong cõi người ta,\n"
for temp in (0.3, 0.8, 1.4):
    out = generate("lstm", LM["lstm"], prompt, 300, temp, np.random.default_rng(1)).split("\n")[:4]
    print(f"── LSTM, temperature {num(temp, 1)}")
    for ln in out:
        print(f"  {ln:<44} {len(syllables(ln))} syllables  {tone_text(syllables(ln))}")

A low temperature gives well-formed lines but tends to repeat familiar phrases; a high temperature breaks the
spelling and the rhythm. Try the GRU and the RNN too (`"gru"`, `"rnn"`), or type an opening line of your own.

## 17 · Limits and next steps

Gates solve part of the gradient problem, but RNNs have two more limits that gates cannot fix:

- **A memory of fixed size.** Everything the network knows about the past has to be compressed into $h$ (and $c$):
  128 numbers, whether it has read one line or the whole poem. Measuring LSTM language models, Khandelwal et al.
  (2018) found that the model uses about 200 tokens of context, but for anything farther back than about 50 tokens it
  nearly ignores word order, keeping only a general sense of the topic.
- **Sequential computation.** Step $t$ needs $h_{t-1}$, so the steps cannot be computed in parallel: reading the
  8,000 validation characters takes 8,000 computations one after another, however many cores the machine has.

In machine translation, the first limit becomes a bottleneck: the RNN encoder has to squeeze the whole source
sentence into one vector, and translation quality drops as sentences get longer. The solution is to let the decoder
**look back** at every state of the encoder at each step, instead of relying only on the final vector: the attention
mechanism. Going further, the Transformer drops the loop entirely, so it can compute every position at once. The
forget-gate idea does not go away: it returns in linear models such as Mamba in
[Lesson 25](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn-tro-lai/).

## Summary
- An RNN uses **the same set of weights** at every step: $h_t = \tanh(x_tW_x + h_{t-1}W_h + b)$. BPTT is
  backpropagation on the network unrolled in time, and the gradient of $W_h$ is a sum over every step.
- At each step back, the gradient is multiplied by $W_h^\top$ and $1 - h_t^2$: at initialization, a 32-unit RNN gets
  $3.7 \cdot 10^{-8}$ of the gradient back across 80 steps; with $g = 2$ it is the opposite: it explodes.
- Along the LSTM's conveyor belt, $\partial c_t/\partial c_{t-1} = f_t$. But with $b_f = 0$, $f_t \approx 0.5$, and
  the LSTM keeps even less gradient than the RNN; $b_f = 3$ (or $b_z = 3$ for the GRU) keeps 0.2 (0.06 for the GRU)
  after 80 steps.
- The “remember the first symbol” task (3 seeds, 5,000 steps): the RNN and the LSTM with $b_f \le 1$ fail from
  $T = 40$ on; $b_f = 3$ and $b_z = 3$ remember up to $T = 160$ in 29/30 runs. Initialization matters as much as
  architecture.
- BPTT truncated at $k = 10$ steps: the RNN cannot learn a 20-step dependency.
- *The Tale of Kiều*, character level: LSTM 2.2637, GRU 2.3412, RNN 2.4595 bits per character (seed 0). All three
  learn the level/oblique tone rule.
- The most robust test is the choice to end the line after syllable 6: every RNN seed 52.4–57.9% (random guessing:
  50%), including the 280-unit RNN; 5/9 gated seeds clearly beat them, the best reaching 89.4%. LSTM and GRU differ
  by no more than the variation between seeds.

## Review questions
1. Why doesn't the number of parameters of an RNN depend on the length of the sequence? What does that imply for the
   gradient of $W_h$?
2. What does the gradient get multiplied by as it flows back through one step of an RNN? Why does gradient clipping
   fix exploding gradients but not vanishing ones?
3. LSTM has a memory-cell conveyor belt, and yet with $b_f = 0$ it keeps the gradient worse than a plain RNN. Explain
   why, and say how to fix it.
4. With BPTT truncated at $k = 10$, the RNN cannot learn the $T = 20$ task, but the LSTM with $b_f = 3$ reaches about
   78%. Why?
5. The GRU's seed 1 alternates 6↔8 better than every LSTM seed. Can we conclude that the GRU is better than the LSTM?
   Which measurement is more trustworthy, and why?
6. The RNN with 280 hidden units has as many parameters as the LSTM, but still chooses right after syllable 6 only
   at close to random guessing. What does that say about an RNN's memory?

<details><summary>Hints and answers</summary>

1. Every step shares $W_x$, $W_h$, $b$. So the gradient of $W_h$ is the sum of the contributions
   $h_{t-1}^\top\delta_t$ of every step, and the contributions of distant steps have been multiplied by many factors
   on their way back.
2. By $W_h^\top$ and by the tanh derivative $1 - h_t^2 \le 1$. Clipping shrinks a gradient that is too large down to
   the allowed size while keeping its direction; but once the gradient is as small as $10^{-8}$, the part coming from
   distant steps has sunk beneath the part from nearby steps, and scaling up the whole vector does not bring it back
   to the surface.
3. With $b_f = 0$, $f_t \approx \sigma(0) = 0.5$, so at each step the conveyor belt keeps half:
   $0.5^{80} \approx 10^{-24}$; the other paths go through $W_h$ and add sigmoid factors. Fix: add 1 or more to $b_f$
   at initialization.
4. With $f_t \approx 0.95$ from the start, the untrained memory cell already carries part of the first symbol to the
   end of the sequence; the readout layer only needs the gradient at the last step to learn to exploit it. Truncation
   blocks learning how to remember, not the memory that is already there.
5. No. The three seeds of each network vary enormously (GRU: 0.1 to 28.7 points), and the difference between LSTM and
   GRU lies within that variation. The line-break test on real lines is more trustworthy because it has no sampling
   noise, and it clearly separates the RNN (every seed close to 50%) from the gated networks.
6. More hidden units do not help the gradient cross the roughly 26 characters between the previous line's punctuation
   mark and the decision point; without a shortcut like the conveyor belt, the network cannot learn long-range
   memory, however big its state.
</details>

## Further reading
- J. Elman (1990), [Finding Structure in Time](https://doi.org/10.1207/s15516709cog1402_1): the simple recurrent
  network.
- S. Hochreiter & J. Schmidhuber (1997), [Long Short-Term Memory](https://doi.org/10.1162/neco.1997.9.8.1735);
  F. Gers, J. Schmidhuber & F. Cummins (2000), [Learning to Forget](https://doi.org/10.1162/089976600300015015): the
  forget gate.
- R. Pascanu, T. Mikolov & Y. Bengio (2013), [On the difficulty of training recurrent neural networks](https://arxiv.org/abs/1211.5063):
  the conditions for vanishing and exploding gradients, gradient clipping.
- K. Cho et al. (2014), [Learning Phrase Representations using RNN Encoder–Decoder](https://arxiv.org/abs/1406.1078)
  (§2.3: the gated unit); J. Chung et al. (2014), [Empirical Evaluation of Gated Recurrent Neural Networks](https://arxiv.org/abs/1412.3555).
- R. Jozefowicz, W. Zaremba & I. Sutskever (2015), [An Empirical Exploration of Recurrent Network Architectures](https://proceedings.mlr.press/v37/jozefowicz15.html):
  adding 1 to the forget-gate bias.
- A. Karpathy, J. Johnson & L. Fei-Fei (2015), [Visualizing and Understanding Recurrent Networks](https://arxiv.org/abs/1506.02078):
  memory cells that track line length and quotation marks.
- U. Khandelwal et al. (2018), [Sharp Nearby, Fuzzy Far Away](https://arxiv.org/abs/1805.04623): how much context an
  LSTM uses.
- C. Olah (2015), [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/);
  A. Karpathy (2015), [The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/).
- d2l.ai, [Backpropagation Through Time](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html): full BPTT,
  regular truncation and randomized truncation.
- Nguyễn Du, [Truyện Kiều (*The Tale of Kiều*)](https://vi.wikisource.org/wiki/Truy%E1%BB%87n_Ki%E1%BB%81u) on the
  Vietnamese Wikisource (revision 205132).

**Next lesson:** [Lesson 16 · The attention mechanism](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/attention/):
instead of compressing the whole sentence into one state, look back at exactly the right place.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")